In [1]:
from get_shot_charts import get_all_player_shot_charts
from utils import open_dataframe_in_temp_excel


In [2]:
# player_id = dm.get_player_nba_id(player)
# shot_chart = get_shot_chart_data(player_id=player_id, season="2024-25")
# print(shot_chart.head())

In [3]:
# from utils import open_dataframe_in_temp_excel

# open_dataframe_in_temp_excel(shot_chart)

In [4]:
# import numpy as np

# shot_chart['DISTANCE'] = np.sqrt(shot_chart['LOC_X']**2 + shot_chart['LOC_Y']**2)
# features = shot_chart[['LOC_X', 'LOC_Y', 'DISTANCE']].to_numpy()



In [5]:
# display(features)

In [6]:

# hdb = hdbscan.HDBSCAN(min_samples=5, min_cluster_size=10)

# shot_chart['Cluster'] = hdb.fit_predict(features)



In [7]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 8))
# sns.scatterplot(
#     x='LOC_X',
#     y='LOC_Y',
#     hue='Cluster',
#     palette='viridis',
#     data=shot_chart,
#     legend='full'
# )
# plt.title('Shot Data Clustering')
# plt.xlabel('LOC_X')
# plt.ylabel('LOC_Y')
# plt.axhline(0, color='gray', linestyle='--', alpha=0.5)  # Mark basket
# plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
# plt.show()


In [8]:
shot_charts = get_all_player_shot_charts()

Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
Got the player ID
Got the shot chart
G

e:\coding_projects\nba_01\get_shot_charts.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_shot_charts = pd.concat(player_shot_charts, ignore_index=True)


In [9]:
for chart in shot_charts:
    display(chart)

'PLAYER_NAME'

'TEAM_NAME'

'LOC_X'

'LOC_Y'

'SHOT_ZONE_BASIC'

...

'SHOT_ZONE_RANGE'

'SHOT_DISTANCE'

'SHOT_ATTEMPTED_FLAG'

'SHOT_MADE_FLAG'

'DISTANCE'

In [10]:
def aggregate_shot_data(all_shot_charts):
    # Group by PLAYER_NAME and shot zone
    aggregated_data = (
        all_shot_charts.groupby(['PLAYER_NAME', 'SHOT_ZONE_BASIC'])
        .agg(
            Shot_Count=('SHOT_ATTEMPTED_FLAG', 'sum'),
            Made_Shots=('SHOT_MADE_FLAG', 'sum')
        )
        .reset_index()
    )

    # Calculate Shot Accuracy
    aggregated_data['Shot_Accuracy'] = aggregated_data['Made_Shots'] / aggregated_data['Shot_Count']
    return aggregated_data


In [11]:
display(shot_charts.columns)

Index(['PLAYER_NAME', 'TEAM_NAME', 'LOC_X', 'LOC_Y', 'SHOT_ZONE_BASIC',
       'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE',
       'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'DISTANCE'],
      dtype='object')

In [50]:
def calculate_shot_distributions(aggregated_data):
    # Calculate total shots per player
    total_shots = aggregated_data.groupby('PLAYER_NAME')['Shot_Count'].transform('sum')

    # Normalize shot counts to get distributions
    aggregated_data['Shot_Distribution'] = aggregated_data['Shot_Count'] / total_shots
    return aggregated_data

In [51]:
agg_data = aggregate_shot_data(shot_charts)
agg_data = calculate_shot_distributions(agg_data)

In [54]:
display(agg_data)

,PLAYER_NAME,SHOT_ZONE_BASIC,Shot_Count,Made_Shots,Shot_Accuracy,Shot_Distribution
0,AJ Green,Above the Break 3,89,35,0.393258,0.635714
1,AJ Green,Backcourt,1,1,1.0,0.007143
2,AJ Green,In The Paint (Non-RA),3,1,0.333333,0.021429
3,AJ Green,Left Corner 3,17,10,0.588235,0.121429
4,AJ Green,Mid-Range,8,5,0.625,0.057143
...,...,...,...,...,...,...
2673,Zion Williamson,Above the Break 3,5,2,0.4,0.048077
2674,Zion Williamson,Backcourt,1,0,0.0,0.009615
2675,Zion Williamson,In The Paint (Non-RA),41,14,0.341463,0.394231
2676,Zion Williamson,Mid-Range,2,0,0.0,0.019231


In [55]:
def prepare_feature_matrix(aggregated_data):
    # Pivot data to create a feature matrix
    feature_matrix = aggregated_data.pivot(
        index='PLAYER_NAME',
        columns='SHOT_ZONE_BASIC',
        values='Shot_Distribution'
    ).fillna(0)  # Fill missing values with 0

    # Reset index for clustering
    feature_matrix.reset_index(inplace=True)
    return feature_matrix


In [56]:
feature_matrix = prepare_feature_matrix(agg_data)

C:\Users\rusta\AppData\Local\Temp\ipykernel_24880\3801274984.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)  # Fill missing values with 0


In [57]:
display(feature_matrix)

SHOT_ZONE_BASIC,PLAYER_NAME,Above the Break 3,Backcourt,In The Paint (Non-RA),Left Corner 3,Mid-Range,Restricted Area,Right Corner 3
0,AJ Green,0.635714,0.007143,0.021429,0.121429,0.057143,0.014286,0.142857
1,AJ Johnson,0.000000,0.000000,0.166667,0.000000,0.000000,0.666667,0.166667
2,Aaron Gordon,0.217054,0.000000,0.085271,0.062016,0.077519,0.511628,0.046512
3,Aaron Holiday,0.607843,0.000000,0.137255,0.078431,0.078431,0.039216,0.058824
4,Aaron Nesmith,0.222222,0.000000,0.138889,0.027778,0.083333,0.472222,0.055556
...,...,...,...,...,...,...,...,...
493,Zach Edey,0.084746,0.000000,0.279661,0.008475,0.033898,0.584746,0.008475
494,Zach LaVine,0.397368,0.000000,0.057895,0.039474,0.176316,0.281579,0.047368
495,Zeke Nnaji,0.217391,0.000000,0.130435,0.086957,0.043478,0.478261,0.043478
496,Ziaire Williams,0.201258,0.012579,0.119497,0.050314,0.031447,0.433962,0.150943


In [74]:
from sklearn.preprocessing import StandardScaler
import hdbscan

def cluster_players(feature_matrix):
    # Normalize the feature matrix
    scaler = StandardScaler()
    features = scaler.fit_transform(feature_matrix.drop(columns=['PLAYER_NAME']))

    # Apply HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_samples=3, min_cluster_size=5)
    clusters = clusterer.fit_predict(features)

    # Add cluster labels to the feature matrix
    feature_matrix['Cluster'] = clusters
    return feature_matrix, clusterer


In [79]:
feature_matrix, model = cluster_players(feature_matrix)

c:\Users\rusta\.virtualenvs\nba_01-7yhASJUA\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\rusta\.virtualenvs\nba_01-7yhASJUA\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [82]:
display(feature_matrix.columns)

Index(['PLAYER_NAME', 'Above the Break 3', 'Backcourt',
       'In The Paint (Non-RA)', 'Left Corner 3', 'Mid-Range',
       'Restricted Area', 'Right Corner 3', 'Cluster'],
      dtype='object', name='SHOT_ZONE_BASIC')

In [ ]:
import pandas as pd
threes = feature_matrix['Above the Break 3'] + feature_matrix['Left Corner 3'] + feature_matrix['Right Corner 3']
short_twos = feature_matrix['In The Paint (Non-RA)'] + feature_matrix['Backcourt'] + feature_matrix['Restricted Area']
mid_range = feature_matrix['Mid-Range']

new_features = pd.DataFrame()

In [80]:
clusters = feature_matrix['Cluster'].value_counts()
display(clusters)

Cluster
 5    430
-1     35
 2      9
 0      8
 3      6
 4      5
 1      5
Name: count, dtype: int64

In [77]:
# Group players by their cluster
players_by_cluster = feature_matrix.groupby('Cluster')['PLAYER_NAME'].apply(list).reset_index()

# Rename columns for readability
players_by_cluster.columns = ['Cluster', 'Players']


In [78]:
from utils import open_dataframe_in_temp_excel
open_dataframe_in_temp_excel(feature_matrix)

Temporary Excel file created: C:\Users\rusta\AppData\Local\Temp\tmp80te_pm6.xlsx
Opening Excel...
Excel opened successfully.
Excel closed.
Could not delete the file. Ensure Excel is fully closed.
